# Qwen3.5-9B Setup & Inference on Kaggle (GPU Optimized)

> **Prerequisites in Kaggle Notebook Settings (Right Sidebar):**
- **Accelerator:** GPU T4 x 2 (or GPU P100)
- **Internet:** ON (Required to download model weights from Hugging Face)

In [ ]:
# Step 1: Install latest high-efficiency dependencies
!pip install -q -U transformers accelerate bitsandbytes torch

In [ ]:
# Step 2: Verify GPU availability
import torch

gpu_count = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Detected GPUs: {gpu_count}")
for i in range(gpu_count):
    print(f"  Device {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB)")

In [ ]:
# Step 3: Load Qwen3.5-9B-Instruct with auto device mapping & 4-bit / bfloat16 optimization
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen3.5-9B-Instruct"

# 4-bit quantization config to run fast within free Kaggle VRAM limits
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

print("Loading Model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("Model loaded successfully!")

In [ ]:
# Step 4: Run Inference Test
prompt = "မင်္ဂလာပါ Qwen3.5-9B ရေ၊ မင်းရဲ့ စွမ်းဆောင်ရည်တွေကို အကျဉ်းချုပ် ရှင်းပြပေးပါ။"
messages = [
    {"role": "system", "content": "You are a helpful and knowledgeable AI assistant."},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

with torch.no_grad():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9
    )

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("\n=== Response ===\n")
print(response)